# MLP Agents (Percentile Data)

This notebook trains the insider threat agents using a Multi-Layer Perceptron (MLP) on percentile temporal representations. This replaces the need for sequential memory (like LSTM) since the temporal history is already captured by the percentile features.

## 1. Data Pipeline

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler


# === Determinism: seed all RNGs (same config as week3/tier1_score_generation.ipynb) ===
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Use a seeded generator for DataLoader shuffle
g = torch.Generator()
g.manual_seed(SEED)

print(f"All RNGs seeded with SEED={SEED}")

# ============================================================================
# Fixed 60/20/20 user-level percentile splits (single source of truth)
# ============================================================================
# The pre-split percentile pickles are produced by
# week3/percentile_mlp_user_split.ipynb (which replicates the stratified
# user-level 60/20/20 split + SEED=42 from week3/tier1_score_generation.ipynb)
# and saved to data/user_split_formlp/. This notebook loads them directly and
# never re-splits users.

def _find_project_root():
    """Walk up to pyproject.toml (project portability convention)."""
    _curr = Path.cwd()
    while not (_curr / 'pyproject.toml').exists() and _curr != _curr.parent:
        _curr = _curr.parent
    return _curr

def load_pre_split_data(split_dir=None):
    """Load the three pre-split percentile DataFrames from data/user_split_formlp/.

    Returns (train_df, val_df, test_df) and verifies zero user overlap.
    """
    if split_dir is None:
        split_dir = _find_project_root() / "data" / "user_split_formlp"
    split_dir = Path(split_dir)
    files = {
        "train": split_dir / "dayr4.2-percentile30-train.pkl",
        "val": split_dir / "dayr4.2-percentile30-validation.pkl",
        "test": split_dir / "dayr4.2-percentile30-test.pkl",
    }
    for key, path in files.items():
        if not path.exists():
            raise FileNotFoundError(
                f"Split file not found: {path}. Run "
                f"week3/percentile_mlp_user_split.ipynb first."
            )
    train_df = pd.read_pickle(str(files["train"]))
    val_df = pd.read_pickle(str(files["val"]))
    test_df = pd.read_pickle(str(files["test"]))

    for dfp in (train_df, val_df, test_df):
        for col in dfp.select_dtypes(include=['float64']).columns:
            dfp[col] = dfp[col].astype('float32')

    train_set = set(train_df['user'].unique())
    val_set = set(val_df['user'].unique())
    test_set = set(test_df['user'].unique())
    assert not (train_set & val_set), "User leakage: train ∩ validation"
    assert not (train_set & test_set), "User leakage: train ∩ test"
    assert not (val_set & test_set), "User leakage: validation ∩ test"

    n = len(train_set) + len(val_set) + len(test_set)
    print("=" * 60)
    print("OFFICIAL 60/20/20 USER SPLIT (data/user_split_formlp/)")
    print("=" * 60)
    print(f"Total users:       {n}")
    print(f"Training users:    {len(train_set)} ({100*len(train_set)/n:.0f}%)")
    print(f"Validation users:  {len(val_set)} ({100*len(val_set)/n:.0f}%)")
    print(f"Testing users:     {len(test_set)} ({100*len(test_set)/n:.0f}%)")
    print(f"Train ∩ Validation: {len(train_set & val_set)}")
    print(f"Train ∩ Test:       {len(train_set & test_set)}")
    print(f"Validation ∩ Test:  {len(val_set & test_set)}\n")
    return train_df, val_df, test_df

class ModalityDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

def get_modality_columns(all_cols, modality):
    """Deterministic, ordered feature columns (sorted, no set()) — mirrors week3/tier1_lib."""
    personality = ['O', 'C', 'E', 'A', 'N']
    if modality == 'logon':
        cols = [c for c in all_cols if 'logon' in c or 'logout' in c]
    elif modality == 'email':
        cols = [c for c in all_cols if 'email' in c or 'mail' in c]
    elif modality == 'http':
        cols = [c for c in all_cols if 'http' in c or 'url' in c]
    elif modality == 'file_device':
        keywords = ['file', 'device', 'comp', 'disk', 'phof', 'docf', 'txtf', 'exef']
        cols = [c for c in all_cols if any(k in c for k in keywords)]
    else:
        raise ValueError(f"Unknown modality: {modality}")
    return sorted(cols) + personality

def get_data_loaders(data_path, modality, split_dir=None, batch_size=128):
    """Load pre-split train/val/test pickles from data/user_split_formlp/ and
    build train (undersampled), validation and test loaders."""
    print(f"Using pre-split percentile data for modality: {modality}")
    train_df, val_df, test_df = load_pre_split_data(split_dir=split_dir)

    feature_cols = get_modality_columns(train_df.columns.tolist(), modality)
    print(f"Selected {len(feature_cols)} features for {modality}.")

    y_train = (train_df['insider'].values > 0).astype(int)
    y_val = (val_df['insider'].values > 0).astype(int)
    y_test = (test_df['insider'].values > 0).astype(int)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[feature_cols].fillna(0)).astype(np.float32)
    X_val = scaler.transform(val_df[feature_cols].fillna(0)).astype(np.float32)
    X_test = scaler.transform(test_df[feature_cols].fillna(0)).astype(np.float32)

    print("\n============================================================")
    print("DATASET SUMMARY")
    print("============================================================")
    print(f"{'Split':<12}{'Users':>8}{'Samples':>12}{'Positives':>12}")
    print("-" * 44)
    print(f"{'Train':<12}{len(set(train_df['user'].unique())):>8}{len(X_train):>12}{int(y_train.sum()):>12}")
    print(f"{'Validation':<12}{len(set(val_df['user'].unique())):>8}{len(X_val):>12}{int(y_val.sum()):>12}")
    print(f"{'Test':<12}{len(set(test_df['user'].unique())):>8}{len(X_test):>12}{int(y_test.sum()):>12}")
    print("-" * 44 + "\n")

    # Apply Random Undersampling to TRAIN SET ONLY (existing strategy: 4:1)
    rus = RandomUnderSampler(sampling_strategy=0.25, random_state=42)
    X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

    print(f"Resampled training samples: {len(X_train_resampled)} | Positives: {y_train_resampled.sum()}")

    train_dataset = ModalityDataset(X_train_resampled, y_train_resampled)
    val_dataset = ModalityDataset(X_val, y_val)
    test_dataset = ModalityDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, len(feature_cols)


All RNGs seeded with SEED=42


## Load Official 60/20/20 Split (data/user_split_formlp/)

This notebook loads the **pre-split train / validation / test percentile pickles** from `data/user_split_formlp/`.

## 2. Model Architecture

In [2]:
class ModalityFFNN(nn.Module):
    """
    A simple Feed-Forward network for flat percentile features.
    """
    def __init__(self, input_dim, hidden_dim=64, dropout=0.2):
        super(ModalityFFNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        out = self.net(x)
        return out.squeeze(1)


## 3. Training Function

In [3]:
import argparse
import os
from pathlib import Path
import mlflow
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import recall_score, confusion_matrix, roc_auc_score
import time

# Resolve project root
_curr = Path.cwd()
while not (_curr / 'pyproject.toml').exists() and _curr != _curr.parent:
    _curr = _curr.parent
PROJECT_ROOT = _curr
DATA_DIR = PROJECT_ROOT / "data"
MLFLOW_DB = PROJECT_ROOT / "mlflow.db"


def _evaluate_loader(model, loader, device):
    """Evaluate a DataLoader with the existing binary metrics (recall/FPR/AUC/CM)."""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).int()

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    recall = recall_score(all_labels, all_preds, zero_division=0)
    roc_auc = roc_auc_score(all_labels, all_probs)
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    return recall, fpr, roc_auc, int(tn), int(fp), int(fn), int(tp), len(all_labels)


def train(modality, data_path=None, epochs=20, lr=0.001, batch_size=128, hidden_dim=64):
    if data_path is None:
        data_path = str(PROJECT_ROOT / "data" / "dayr4.2-percentile30.pkl")
        if not os.path.exists(data_path):
            print(f"Data file {data_path} not found. Please run preprocessing scripts first.")
            return

    # Robust device selection: probe CUDA with a real op, fall back to CPU.
    def _cuda_usable():
        if not torch.cuda.is_available():
            return False
        try:
            x = torch.randn(4, 4, device='cuda')
            _ = x @ x
            torch.cuda.synchronize()
            return True
        except Exception:
            return False

    device = torch.device('cuda' if _cuda_usable() else 'cpu')
    print(f"Training {modality} agent on device: {device}")

    # 1. Setup MLflow
    mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
    mlflow.set_experiment("Week2_MLP_Percentile_Agents")

    with mlflow.start_run(run_name=f"{modality}_mlp_agent"):
        mlflow.log_params({
            "modality": modality,
            "epochs": epochs,
            "learning_rate": lr,
            "batch_size": batch_size,
            "hidden_dim": hidden_dim,
            "model_type": "MLP",
            "balancing": "RandomUnderSampler (Normal:Anomaly 4:1)",
            "split_type": "official_60_20_20_user_split",
            "dataset_source": "user_split_formlp",
            "train_ratio": 0.60,
            "validation_ratio": 0.20,
            "test_ratio": 0.20
        })

        train_loader, val_loader, test_loader, input_dim = get_data_loaders(
            data_path=data_path,
            modality=modality,
            batch_size=batch_size
        )

        mlflow.log_param("input_dim", input_dim)

        _trdf, _vadf, _tedf = load_pre_split_data()
        mlflow.log_params({
            "train_user_count": int(len(set(_trdf['user'].unique()))),
            "validation_user_count": int(len(set(_vadf['user'].unique()))),
            "test_user_count": int(len(set(_tedf['user'].unique()))),
            "random_seed": 42
        })

        if input_dim <= 5:
            print(f"Warning: Only {input_dim} features found for {modality}. Check column matching.")

        model = ModalityFFNN(input_dim=input_dim, hidden_dim=hidden_dim).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(epochs):
            model.train()
            total_loss = 0

            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)

                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")
                mlflow.log_metric("train_loss", avg_loss, step=epoch)

        val_recall, val_fpr, val_auc, val_tn, val_fp, val_fn, val_tp, val_n = _evaluate_loader(model, val_loader, device)
        test_recall, test_fpr, test_auc, test_tn, test_fp, test_fn, test_tp, test_n = _evaluate_loader(model, test_loader, device)

        print(f"--- {modality.upper()} AGENT EVALUATION (VALIDATION) ---")
        print(f"Recall (TPR): {val_recall:.4f}")
        print(f"False Positive Rate (FPR): {val_fpr:.4f}")
        print(f"ROC-AUC: {val_auc:.4f}")
        print(f"Confusion Matrix:\nTN={val_tn} FP={val_fp}\nFN={val_fn} TP={val_tp}")

        print(f"--- {modality.upper()} AGENT EVALUATION (TEST) ---")
        print(f"Recall (TPR): {test_recall:.4f}")
        print(f"False Positive Rate (FPR): {test_fpr:.4f}")
        print(f"ROC-AUC: {test_auc:.4f}")
        print(f"Confusion Matrix:\nTN={test_tn} FP={test_fp}\nFN={test_fn} TP={test_tp}")

        print("\n" + "=" * 60)
        print(f"{modality.upper()} — VALIDATION vs TEST (official 60/20/20 split)")
        print("=" * 60)
        print(f"{'Dataset':<12}{'Samples':>9}{'Recall':>9}{'FPR':>9}{'AUC':>9}{'TN':>7}{'FP':>7}{'FN':>7}{'TP':>7}")
        print("-" * 68)
        print(f"{'Validation':<12}{val_n:>9}{val_recall:>9.4f}{val_fpr:>9.4f}{val_auc:>9.4f}{val_tn:>7}{val_fp:>7}{val_fn:>7}{val_tp:>7}")
        print(f"{'Test':<12}{test_n:>9}{test_recall:>9.4f}{test_fpr:>9.4f}{test_auc:>9.4f}{test_tn:>7}{test_fp:>7}{test_fn:>7}{test_tp:>7}")
        print("=" * 60 + "\n")

        mlflow.log_metrics({
            "val_recall": val_recall,
            "val_fpr": val_fpr,
            "val_roc_auc": val_auc,
            "val_true_positives": val_tp,
            "val_false_positives": val_fp,
            "val_true_negatives": val_tn,
            "val_false_negatives": val_fn,
            "test_recall": test_recall,
            "test_fpr": test_fpr,
            "test_roc_auc": test_auc,
            "true_positives": test_tp,
            "false_positives": test_fp,
            "true_negatives": test_tn,
            "false_negatives": test_fn
        })

        save_dir = Path.cwd() / "saved_models"
        save_dir.mkdir(exist_ok=True)
        model_path = save_dir / f"{modality}_mlp_agent.pt"
        torch.save(model.state_dict(), model_path)
        mlflow.log_artifact(str(model_path))

        print(f"Finished {modality}. Run logged to MLflow.\n")


## 4. Run Training

In [4]:
modalities = ['logon', 'email', 'file_device', 'http']
for mod in modalities:
    print('='*50)
    print(f'STARTING PIPELINE FOR {mod.upper()} AGENT')
    print('='*50)
    train(mod, epochs=20)


STARTING PIPELINE FOR LOGON AGENT
Training logon agent on device: cpu


C:\insider_threat_detection_project2\.venv\Lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


Using pre-split percentile data for modality: logon
OFFICIAL 60/20/20 USER SPLIT (data/user_split_formlp/)
Total users:       1000
Training users:    600 (60%)
Validation users:  200 (20%)
Testing users:     200 (20%)
Train ∩ Validation: 0
Train ∩ Test:       0
Validation ∩ Test:  0

Selected 20 features for logon.

DATASET SUMMARY
Split          Users     Samples   Positives
--------------------------------------------
Train            600      185744         574
Validation       200       61076         191
Test             200       61203         201
--------------------------------------------

Resampled training samples: 2870 | Positives: 574
OFFICIAL 60/20/20 USER SPLIT (data/user_split_formlp/)
Total users:       1000
Training users:    600 (60%)
Validation users:  200 (20%)
Testing users:     200 (20%)
Train ∩ Validation: 0
Train ∩ Test:       0
Validation ∩ Test:  0

Epoch 1/20 | Loss: 0.6250
Epoch 5/20 | Loss: 0.4295
Epoch 10/20 | Loss: 0.3895
Epoch 15/20 | Loss: 0.3519
Epoch 